# MS/MS 峰 → MAGMa 候选碎片标注

原算法，无训练；原子索引以 engine_smiles 为准并逐碎片验证，标注不等于实验确认的机制。

本 notebook 展示已记录的真实评测结果。模型推理需显式开启，始终不进行训练。

In [1]:
import json, os, sys, subprocess, tempfile
from pathlib import Path
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'openmolspec_bench').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from the cloned OpenMolSpec frontend repository')
task_id = 'ms.annotation.magma'
example_input = {'spec_id': 'example-input-only', 'smiles': 'CCO', 'adduct': '[M+H]+', 'collision_energy': 20, 'instrument': 'Orbitrap', 'precursor_mz': 47.0491, 'peaks': [[31.0178, 1.0], [47.0491, 0.4]]}
print(json.dumps(example_input, indent=2, ensure_ascii=False))

{
  "spec_id": "example-input-only",
  "smiles": "CCO",
  "adduct": "[M+H]+",
  "collision_energy": 20,
  "instrument": "Orbitrap",
  "precursor_mz": 47.0491,
  "peaks": [
    [
      31.0178,
      1.0
    ],
    [
      47.0491,
      0.4
    ]
  ]
}


## 已记录的评测

保留样本量、输入条件和来源文件；流程执行成功不等于精度达标。

In [2]:
index = json.loads((root / 'documentation/assets/evaluation_index.json').read_text())
recorded = index.get(task_id, [])
print(json.dumps(recorded or {'status': 'not_evaluated', 'reason': 'Required compatible assets or adapter unavailable'}, indent=2, ensure_ascii=False))

[
  {
    "task": "ms.annotation.magma",
    "metrics": {
      "matched_peak_fraction": 0.5681818181818182,
      "annotated_spectra": 2,
      "checked_fragments": 1534
    },
    "counts": {
      "passed": 2
    },
    "selection": {
      "split_rows_examined": 2040,
      "exceeds_heavy_atom_limit": 2005,
      "duplicate_molecule": 33,
      "missing_acquisition_metadata": 0
    },
    "source_file": "magma_real_test_verified_20261115.json",
    "sample_limit": 2
  }
]


## 可选：冻结权重推理

设置 `OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE=1` 并配置工作区和模型环境。缺少权重时跳过或报错，不会启动训练。

In [3]:
if os.environ.get('OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE') == '1':
    runtime = os.environ.get('OPENMOLSPEC_PYTHON', sys.executable)
    workspace = os.environ.get('OPENMOLSPEC_BENCH_ROOT', str(root.parent))
    with tempfile.TemporaryDirectory(prefix='openmolspec-notebook-') as directory:
        command = [runtime, '-m', 'openmolspec_bench', 'run', '--workspace', workspace, '--tasks', task_id, '--limit', '2', '--output', str(Path(directory) / 'result.json')]
        subprocess.run(command, cwd=root, check=True)
        print((Path(directory) / 'result.json').read_text())
else:
    print('Model execution disabled; recorded evaluation displayed above. No training performed.')

Model execution disabled; recorded evaluation displayed above. No training performed.
